# 01 — Preprocess Audio

Convert all `.m4a` recordings to 16 kHz mono WAV files (20 s, zero-padded).

In [ ]:
import sys
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

sys.path.insert(0, str(Path("..").resolve()))

from config import DATA_DIR, TARGET_SR, TARGET_DURATION

In [ ]:
extensions = {".m4a", ".mp3", ".flac", ".ogg", ".aac"}
target_samples = int(TARGET_DURATION * TARGET_SR)

for sentence_dir in sorted(DATA_DIR.iterdir()):
    if not sentence_dir.is_dir() or not sentence_dir.name.isdigit():
        continue

    text_file = sentence_dir / "text"
    if text_file.exists():
        print(f"Sentence {sentence_dir.name}: {text_file.read_text().strip()[:80]}...")

    for audio_file in sorted(sentence_dir.iterdir()):
        if audio_file.suffix.lower() not in extensions:
            continue

        y, sr = librosa.load(str(audio_file), sr=TARGET_SR, mono=True)
        if len(y) < target_samples:
            y = np.pad(y, (0, target_samples - len(y)))
        else:
            y = y[:target_samples]

        wav_path = audio_file.with_suffix(".wav")
        sf.write(str(wav_path), y, TARGET_SR)
        print(f"  {audio_file.name} -> {wav_path.name}  ({len(y)/TARGET_SR:.2f}s)")